# One-Shot Evaluation of OTEL Traces with Weave

Run an eval directly on OTEL-instrumented traces — no need to publish a dataset first.  
Traces are fetched, scored, and evaluated in a single pass, just like a native Weave offline evaluation.

In [ ]:
import json, os, dotenv, openai, weave

dotenv.load_dotenv()

TEAM_NAME = "smle-demo"
PROJECT = "otel-evals"

oai_client = openai.OpenAI()  # uses OPENAI_API_KEY from env
client = weave.init(f"{TEAM_NAME}/{PROJECT}")

### Step 1 — Fetch OTEL traces and build rows in one step

Pull `ChatCompletion` spans and extract input/output directly into a list of dicts.  
No DataFrame, no `weave.publish()` — just plain rows ready for evaluation.

In [ ]:
calls = list(client.get_calls(
    filter={"op_names": [f"weave:///{TEAM_NAME}/{PROJECT}/op/chatcompletion:*"]},
))

rows = []
for c in calls:
    input_text = " ".join(
        m["content"] for m in c.inputs["input.value"]["messages"] if m.get("content")
    )
    output_text = (
        ((c.output or {}).get("output.value") or {})
        .get("choices", [{}])[0]
        .get("message", {})
        .get("content", "")
    )
    if input_text or output_text:
        rows.append({"input": input_text, "output": output_text})

print(f"{len(rows)} rows ready for evaluation")

### Step 2 — Define scorers

Same scorers as before:
- **Sentiment** — LLM-as-judge using `gpt-4o-mini` to label positive / negative / neutral
- **Response length** — word count with a 0–1 adequacy score

In [ ]:
@weave.op(name="sentiment")
def sentiment_scorer(output: str):
    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": 'Respond ONLY with JSON: {"label": "positive"|"negative"|"neutral", "score": 0-1}'},
            {"role": "user",   "content": output[:4000]},
        ],
        max_tokens=60,
    )
    data = json.loads(resp.choices[0].message.content)
    return {"label": data["label"], "score": float(data["score"])}


@weave.op(name="response_length")
def length_scorer(output: str):
    words = len(output.split())
    score = min(1.0, words / 50.0) if words >= 10 else 0.2 * (words / 10.0)
    return {"word_count": words, "score": round(score, 4)}

### Step 3 — Evaluate in one shot

Pass the rows directly to `weave.Evaluation` — it accepts a plain list of dicts, so there's no need to create or publish a separate dataset. The passthrough model returns the trace output as-is; we're only scoring what OTEL already captured.

In [ ]:
@weave.op()
def passthrough_model(output: str):
    return output

evaluation = weave.Evaluation(
    dataset=rows,
    scorers=[sentiment_scorer, length_scorer],
    evaluation_name="otel_traces_eval",
)

result = await evaluation.evaluate(passthrough_model)
print(result)